# 01 — Collect GPT-2 Activations

This notebook collects residual stream activations from GPT-2 small (layer 6)
for subsequent decomposition analysis.

**Analogy to X-ray scattering:** Each forward pass through the model is like an
"exposure" — we collect the resulting activation vectors (analogous to scattering
patterns) from many text samples, building up a dataset for decomposition analysis.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from sae.activations import ActivationCollector, load_activations
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


## Configuration

We collect from layer 6 (middle of the 12-layer GPT-2 small network).
This layer has good feature diversity — early layers tend to represent
more syntactic features, later layers more semantic ones. Layer 6 gives
a good mix.

In [2]:
# Configuration
MODEL_NAME = "gpt2-small"
LAYER = 6
N_TOKENS = 2_000_000  # 2M tokens — enough for good training
SEQ_LEN = 128
BATCH_SIZE = 32
SAVE_PATH = "../results/activations_layer6_2M.pt"

In [3]:
# Initialize collector
collector = ActivationCollector(
    model_name=MODEL_NAME,
    layer=LAYER,
)
collector.load_model()

Loading gpt2-small...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer
Loaded. d_model=768, n_layers=12


In [4]:
# Collect activations
# This will take ~5-10 minutes depending on GPU speed
activations = collector.collect_from_dataset(
    n_tokens=N_TOKENS,
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
    save_path=SAVE_PATH,
)

print(f"\nActivation tensor shape: {activations.shape}")
print(f"Memory: {activations.element_size() * activations.nelement() / 1e9:.2f} GB")

Loading dataset: monology/pile-uncopyrighted...


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Tokens collected: 100%|████████████████████████████████████████████████████████████████████████████████████████| 2000000/2000000 [01:59<00:00, 16746.88it/s]


Collected 2000000 activation vectors of dim 768
Saved to ../results/activations_layer6_2M.pt

Activation tensor shape: torch.Size([2000000, 768])
Memory: 6.14 GB


In [5]:
# Quick sanity check: activation statistics
print(f"Mean: {activations.mean():.4f}")
print(f"Std:  {activations.std():.4f}")
print(f"Min:  {activations.min():.4f}")
print(f"Max:  {activations.max():.4f}")

Mean: 0.0000
Std:  10.2515
Min:  -74.3208
Max:  3036.2224


## Also save the tokens for later interpretation

We need the original tokens to interpret what features mean.

In [6]:
# Re-collect tokens only (for interpretation later)
from datasets import load_dataset

dataset = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)

all_tokens = []
n_sequences = N_TOKENS // SEQ_LEN
token_buffer = []
sequences_collected = 0

for example in dataset:
    text = example.get("text", "")
    if not text:
        continue
    tokens = collector.model.to_tokens(text, prepend_bos=False).squeeze(0)
    token_buffer.extend(tokens.tolist())
    
    while len(token_buffer) >= SEQ_LEN:
        all_tokens.extend(token_buffer[:SEQ_LEN])
        token_buffer = token_buffer[SEQ_LEN:]
        sequences_collected += 1
        if sequences_collected >= n_sequences:
            break
    if sequences_collected >= n_sequences:
        break

tokens_tensor = torch.tensor(all_tokens[:N_TOKENS])
torch.save(tokens_tensor, "../results/tokens_2M.pt")
print(f"Saved {tokens_tensor.shape[0]} tokens")

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Saved 2000000 tokens


---
**Next:** `02_train_sae.ipynb` — Train a Sparse Autoencoder on these activations